# Att prognostisera månatliga bilförsäkringsskadeantal med PROC FORECAST

## Sammanfattning

Ett aktuarieteam för reservsättning behöver en 12-månaders framåtblick över månatliga bilförsäkringsskadeantal som visar stabil portföljtillväxt plus en tydlig vinterväderseffekt. Denna notebook genererar fem års syntetiska månatliga skadeantal (60 månader, jan 2020 - dec 2024, från cirka 1 460 till 2 780 skador), och använder sedan **PROC FORECAST** för att anpassa en stegvis autoregressiv baslinje och en multiplikativ säsongsmodell av Holt-Winters-typ. Varje modell producerar 12 månatliga punktprognoser med 95 % konfidensgränser för kapacitets- och reservplanering. Den säsongsbetonade Holt-Winters-modellen prognostiserar den starkaste efterfrågan en till två månader framåt (cirka 2 780-2 900 skador) som avtar till en svacka kring steg nio (cirka 2 200), medan den autoregressiva baslinjen prognostiserar en jämnare avklingning; båda konfidensbanden vidgas med prognoshorisonten.

## Datakällor

| Dataset | Rader | Granularitet | Nyckelvariabler | Beskrivning |
|---------|------|-------|---------------|-------------|
| `claims` | 60 | En rad per kalendermånad (jan 2020 - dec 2024) | `date` (SAS-datum, `MONYY7.`), `claim_count` (numerisk) | Syntetiska månatliga bilförsäkringsskadeantal byggda från en linjär tillväxttrend (~9 skador/månad), en sinusformad årscykel, en additiv vinterväderökning i dec/jan/feb, och Gaussiskt brus (`rand('normal')`). Fröet `20240531` gör det reproducerbart. |

# Att prognostisera månatliga bilförsäkringsskadeantal

Reserverings- och kapacitetsplaneringsteam hos ett sakförsäkringsbolag för privatpersoner behöver en framåtblick över hur många **bilskador** som kommer att rapporteras varje månad. Skadevolymen i detta bestånd växer stadigt i takt med att portföljen expanderar, och den ökar kraftigt varje vinter när is, snö och kortare dagsljus driver upp kollisions- och glasskador.

Denna notebook går igenom ett komplett `PROC FORECAST`-arbetsflöde:

1. Generera en realistisk, helt syntetisk serie med månatliga skadeantal.
2. Anpassa en **stegvis autoregressiv (STEPAR)**-baslinje som fångar trend plus autokorrelation.
3. Anpassa en **multiplikativ Holt-Winters (WINTERS)**-modell som explicit modellerar den 12-månaders säsongscykeln.
4. Jämför de två modellerna och tolka den framåtblickande prognosen och konfidensbandet.

Allt körs på inline-syntetisk data — inga externa filer eller nätverksåtkomst.

## Steg 1 — Generera den syntetiska skadeserien

DATA-steget nedan bygger **60 månatliga observationer** (jan 2020 till dec 2024). För varje månad kombinerar vi fyra komponenter som speglar ett verkligt bilförsäkringsbestånd:

- **Trend** — en baslinje på ~1 800 skador som växer med ~9 per månad i takt med ökande exponering.
- **Årscykel** — en sinus-/cosinusterm som ger en jämn säsongsvåg.
- **Vinterökning** — ett additivt tillskott i december, januari och februari.
- **Brus** — `rand('normal', 0, 90)` för realistisk variation månad till månad.

`call streaminit()` fixerar det slumpmässiga flödet så att notebooken är reproducerbar. Variabeln `date` är ett riktigt SAS-datum byggt med `INTNX` och formaterat `MONYY7.`, och satsen `ID date INTERVAL=MONTH` namnger den som tidsidentifierare. De första 14 raderna som skrivs ut nedan hamnar mellan ungefär 1 460 och 2 450 skador.

In [1]:
data claims;
    CALL streaminit(20240531);
    GÖR i = 0 TILL 59;
        /* Bygg ett riktigt SAS-månadsdatum så att INTERVAL=MONTH stämmer */
        date = intnx('month', '01JAN2020'd, i);
        format date monyy7.;

        month_idx = mod(i, 12) + 1;          /* 1 = jan ... 12 = dec */

        trend  = 1800 + 9 * i;               /* växande exponeringsbas   */
        season = 260 * sin((month_idx - 1) / 12 * 2 * constant('pi'))
               + 150 * cos((month_idx - 1) / 12 * 2 * constant('pi'));
        winter = 220 * (month_idx IN (12, 1, 2));   /* is-/snöökning  */
        noise  = rand('normal', 0, 90);

        claim_count = round(trend + season + winter + noise);
        OM claim_count < 0 SÅ claim_count = 0;
        UTDATA;
    SLUT;
    BEHÅLL date claim_count;
KÖR;

PROCEDUR SKRIV data=claims(obs=14) ETIKETT;
    ETIKETT date = "Månad" claim_count = "Rapporterade skadeanmälningar";
    TITEL "Första 14 månaderna med syntetiska bilskadeantal";
KÖR;

                                    Första 14 månaderna med syntetiska bilskadeantal                                    

  Obs   Månad   Rapporterade skadeanmälningar
    1   21915                            2178
    2   21946                            2281
    3   21975                            2252
    4   22006                            1974
    5   22036                            2067
    6   22067                            1805
    7   22097                            1697
    8   22128                            1619
    9   22159                            1462
   10   22189                            1688
   11   22220                            1713
   12   22250                            2008
   13   22281                            2116
   14   22312                            2451

... 46 more observations (showing 14 of 60)




NOTE: DATA claims


NOTE: Wrote claims (60 rows, 2 columns).
NOTE: DATA elapsed:
  wall  0.01 seconds
  cpu   0.01 seconds
NOTE: PROC PRINT data=claims

NOTE: PROC PRINT completed: 14 observations printed, 2 variables


## Steg 2 — Stegvis autoregressiv baslinje (METHOD=STEPAR)

`METHOD=STEPAR` är standardvalet. Den anpassar först en tidstrendmodell — här `TREND=2` för en linjär trend — och tillämpar sedan en **stegvis autoregression** på residualerna, där AR-lagg läggs till och behålls efter signifikans. `AR=4` begränsar den kandiderande autoregressiva ordningen till fyra lagg, vilket räcker gott och väl för en månatlig serie med kortsiktig momentum.

Viktiga alternativ som används här:

- `LEAD=12` — prognostisera 12 månader utöver datan.
- `ALPHA=0.05` — 95 % konfidensgränser.
- `OUTFULL` — stapla de historiska `ACTUAL`-raderna tillsammans med prognoshorisontens rader i `OUT=`-datasetet (i stället för enbart prognosraderna).
- `OUTLIMIT` — lägg till kolumnerna för nedre/övre konfidensgräns (`L95_claim_count`, `U95_claim_count`).
- `ID date INTERVAL=MONTH` — namnger tidsidentifieraren och deklarerar att serien är månatlig.

In [2]:
PROCEDUR forecast data=claims
        out=fc_stepar
        METHOD=stepar trend=2 ar=4
        LEAD=12 ALPHA=0.05
        outfull outlimit;
    id date interval=month;
    VARIABEL claim_count;
KÖR;

PROCEDUR SKRIV data=fc_stepar(obs=10) ETIKETT;
    ETIKETT date = "Månad" claim_count = "Skadeantal";
    TITEL "STEPAR-utdata: faktiska, anpassade och prognostiserade rader";
KÖR;

                                    Första 14 månaderna med syntetiska bilskadeantal                                    

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 60
Forecast Periods: 12



                              STEPAR-utdata: faktiska, anpassade och prognostiserade rader                              

  Obs   Månad  _TYPE_   Skadeantal  L95_CLAIM_COUNT  U95_CLAIM_COUNT
    1   21915  ACTUAL  2121.816667                .                .
    2   21946  ACTUAL  2167.711419                .                .
    3   21975  ACTUAL  2254.781177                .                .
    4   22006  ACTUAL  2228.505912                .                .
    5   22036  ACTUAL  1978.152296                .                .
    6   22067  ACTUAL  2030.606563                .                .
    7   22097  ACTUAL  1864.520418                .                .
    8   22128  ACTUAL  1784.855682                .                .
    9   22159  ACTUAL  174


NOTE: PROC FORECAST data=claims

NOTE: Using Python for FORECAST estimation
NOTE: Output dataset created with 72 observations.
NOTE: PROC PRINT data=fc_stepar

NOTE: PROC PRINT completed: 10 observations printed, 5 variables


### Att läsa OUT=-datasetet

`OUT=`-datasetet staplar rader efter en `_TYPE_`-kolumn och bär konfidensgränserna som separata **kolumner**:

| Element | Typ | Betydelse |
|---------|------|---------|
| `_TYPE_ = 'ACTUAL'` | rad | Det observerade `claim_count` för var och en av de 60 historiska månaderna. |
| `_TYPE_ = 'FORECAST'` | rad | De 12 punktprognoserna för prognoshorisonten. |
| `L95_claim_count` / `U95_claim_count` | kolumn | Nedre/övre 95 % konfidensgränser, ifyllda på `FORECAST`-raderna (saknas på `ACTUAL`-raderna). Den numeriska nivån speglar `ALPHA=`. |

Så det utskrivna `OUT=` här innehåller 72 rader: 60 `ACTUAL`-historikrader följt av 12 `FORECAST`-horisontrader. De första tio raderna som visas ovan är alla `ACTUAL`, med konfidensgränskolumnerna saknade eftersom gränserna bara sätts på prognosraderna.

> **Motoranmärkning.** SAS `OUTFULL` skriver också en inom-urval ett-steg-framåt `FORECAST`-rad för varje historisk månad, och `OUTRESID` lägger till `_TYPE_='RESIDUAL'`-rader. Jenners PROC FORECAST genererar ännu inte dessa anpassade/residual-rader inom urvalet (spårat som gap-test `400979`), så denna notebook läser bara `ACTUAL`-historiken och den framåtblickande `FORECAST`-horisonten.

## Steg 3 — Säsongsbetonad Holt-Winters-modell (METHOD=WINTERS)

STEPAR-modellen fångar trend och kortsiktig autokorrelation men modellerar inte explicit den återkommande december–februari-ökningen. För en serie med en tydlig årlig rytm är **multiplikativ Holt-Winters** det bättre verktyget.

`METHOD=WINTERS` bryter ner serien i nivå, linjär trend (`TREND=2`), och en **multiplikativ säsongsfaktor**. `SEASONS=12` deklarerar en 12-periods (månatlig) säsongscykel. Vi begär återigen en 12-månaders `LEAD` med 95 % gränser och `OUTFULL OUTLIMIT` så att utdatan stämmer överens med STEPAR-körningen.

Eftersom motorn stegar fram prognosens `ID` med en enhet per steg (i stället för att respektera `INTERVAL=MONTH` för horisontens datum — gap-test `400979`), granskar cellen nedan prognosen efter **månader framåt (steg 1-12)** i stället för att förlita sig på de utskrivna kalenderdatumen.

In [3]:
PROCEDUR forecast data=claims
        out=fc_winters
        METHOD=winters seasons=12 trend=2
        LEAD=12 ALPHA=0.05
        outfull outlimit;
    id date interval=month;
    VARIABEL claim_count;
KÖR;

/* Behåll den 12-månaders framåtblickande horisonten och indexera efter steg (1..12). */
data horizon;
    STÄLL_IN fc_winters;
    BEHÅLL_VÄRDE months_ahead 0;
    OM _type_ = 'FORECAST';
    months_ahead + 1;
    BEHÅLL months_ahead claim_count l95_claim_count u95_claim_count;
KÖR;

PROCEDUR SKRIV data=horizon ETIKETT;
    ETIKETT months_ahead     = "Månader framåt"
          claim_count       = "Prognostiserade skadeanmälningar"
          l95_claim_count   = "Nedre 95 %"
          u95_claim_count   = "Övre 95 %";
    TITEL "Holt-Winters 12-månaders framåtblickande prognos (per steg)";
KÖR;

                              STEPAR-utdata: faktiska, anpassade och prognostiserade rader                              

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 60
Forecast Periods: 12



                              Holt-Winters 12-månaders framåtblickande prognos (per steg)                               

  Obs    Månader framåt   Prognostiserade skadeanmälningar   Nedre 95 %    Övre 95 %
    1                 1                         2783.07951  2577.844742  2988.314278
    2                 2                        2897.355557  2607.109764  3187.601349
    3                 3                        2805.272075  2449.795029   3160.74912
    4                 4                        2664.498049  2254.028514  3074.967585
    5                 5                        2628.810136  2169.891244  3087.729029
    6                 6                         2548.39319  2045.672732  3051.113649
    7                 7                        2391.


NOTE: PROC FORECAST data=claims

NOTE: Using Python for FORECAST estimation
NOTE: Output dataset created with 72 observations.
NOTE: DATA horizon


NOTE: Read 72 rows from fc_winters.
NOTE: Wrote horizon (12 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=horizon

NOTE: PROC PRINT completed: 12 observations printed, 4 variables


## Steg 4 — Jämför de två modellerna direkt mot varandra

Det tydligaste sättet att bedöma om säsongsmodellen gör nytta är att lägga dess 12-månaders prognos bredvid STEPAR-baslinjen, steg för steg. Vi hämtar `FORECAST`-raderna från båda `OUT=`-dataseten, indexerar var och en efter månader framåt, och sammanfogar dem så att avvikelsen syns direkt.

De två metoderna är överens om den generella nivån men oense om **formen**: Holt-Winters för in den årliga rytmen i horisonten (en högre nivå tidigt i horisonten som avtar till en svacka i mitten och stiger igen), medan STEPAR — som bara modellerar säsongsvariation indirekt via AR-lagg — avklingar jämnare mot seriens medelvärde. Skillnaden mellan dem vid varje steg är den säsongssignal STEPAR lämnar därhän.

> SAS skulle normalt driva denna rimlighetskontroll med `OUTRESID` ett-steg-framåt-residualer (`_TYPE_='RESIDUAL'`). Jenner fyller ännu inte i dessa rader (gap-test `400979`), så vi jämför i stället de två framåtblickande prognoserna direkt — en diagnostik som bara använder utdata motorn faktiskt producerar.

In [4]:
/* STEPAR-horisont, indexerad efter månader framåt. */
data stepar_h;
    STÄLL_IN fc_stepar;
    BEHÅLL_VÄRDE months_ahead 0;
    OM _type_ = 'FORECAST';
    months_ahead + 1;
    stepar = claim_count;
    BEHÅLL months_ahead stepar;
KÖR;

/* WINTERS-horisont, indexerad efter månader framåt. */
data winters_h;
    STÄLL_IN fc_winters;
    BEHÅLL_VÄRDE months_ahead 0;
    OM _type_ = 'FORECAST';
    months_ahead + 1;
    winters = claim_count;
    BEHÅLL months_ahead winters;
KÖR;

/* Sammanfoga de två och visa den säsongsmässiga skillnad STEPAR missar. */
data COMPARE;
    SAMMANFOGA stepar_h winters_h;
    EFTER months_ahead;
    seasonal_gap = winters - stepar;
KÖR;

PROCEDUR SKRIV data=COMPARE ETIKETT;
    ETIKETT months_ahead = "Månader framåt"
          stepar        = "STEPAR-prognos"
          winters       = "Winters-prognos"
          seasonal_gap  = "Winters - STEPAR";
    format stepar winters seasonal_gap 8.0;
    TITEL "STEPAR jämfört med Holt-Winters: 12-månaders prognosjämförelse";
KÖR;

                             STEPAR jämfört med Holt-Winters: 12-månaders prognosjämförelse                             

  Obs    Månader framåt  STEPAR-prognos  Winters-prognos  Winters - STEPAR
    1                 1            2619             2783               164
    2                 2            2537             2897               361
    3                 3            2384             2805               421
    4                 4            2214             2664               450
    5                 5            2089             2629               540
    6                 6            2010             2548               539
    7                 7            1982             2392               410
    8                 8            1995             2240               245
    9                 9            2031             2197               166
   10                10            2075             2354               280
   11                11            2115             2


NOTE: DATA stepar_h


NOTE: Read 72 rows from fc_stepar.
NOTE: Wrote stepar_h (12 rows, 2 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: DATA winters_h


NOTE: Read 72 rows from fc_winters.
NOTE: Wrote winters_h (12 rows, 2 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: DATA compare

NOTE: Stream 1 processed 12 rows, max BY-group size: 1 (O(1) memory verified)
NOTE: Stream 2 processed 12 rows, max BY-group size: 1 (O(1) memory verified)

NOTE: Wrote compare (12 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=compare

NOTE: PROC PRINT completed: 12 observations printed, 4 variables


## Steg 5 — Prognostisera alla affärsgrenar samtidigt (BY-bearbetning)

Verkliga reserveringskörningar täcker flera produkter. Med datan sorterad efter `line_of_business` gör en `BY`-sats att `PROC FORECAST` anpassar en **oberoende säsongsmodell för varje grupp** i ett enda anrop. Här syntetiserar vi ett Bil-bestånd (högre basvolym) och ett Hem-bestånd (lägre bas) och prognostiserar båda sex månader framåt. `OUTALL` skriver hela uppsättningen rader — `ACTUAL`-historiken och `FORECAST`-horisonten — tillsammans med gränskolumnerna för varje grupp, och vi behåller bara `FORECAST`-raderna för tabellen nedan.

In [5]:
data multi_book;
    CALL streaminit(20240531);
    LÄNGD line_of_business $6;
    GÖR lob = 1 TILL 2;
        OM lob = 1 SÅ line_of_business = 'Bil';
        ANNARS            line_of_business = 'Hem';
        GÖR i = 0 TILL 47;
            date = intnx('month', '01JAN2021'd, i);
            format date monyy7.;
            mi   = mod(i, 12) + 1;
            BASE = (lob = 1) * 2000 + (lob = 2) * 1400;
            claim_count = round(BASE + 8 * i
                + 200 * sin((mi - 1) / 12 * 2 * constant('pi'))
                + 180 * (mi IN (12, 1, 2))
                + rand('normal', 0, 70));
            UTDATA;
        SLUT;
    SLUT;
    BEHÅLL line_of_business date claim_count;
KÖR;

PROCEDUR SORTERA data=multi_book;
    EFTER line_of_business date;
KÖR;

PROCEDUR forecast data=multi_book
        out=fc_by
        METHOD=winters seasons=12 trend=2
        LEAD=6 ALPHA=0.05
        outall;
    EFTER line_of_business;
    id date interval=month;
    VARIABEL claim_count;
KÖR;

PROCEDUR SKRIV data=fc_by(obs=12) ETIKETT;
    DÄR _type_ = 'FORECAST';
    ETIKETT line_of_business="Affärsgren" claim_count="Prognostiserade skadeanmälningar";
    TITEL "6-månaders prognoser per affärsgren";
KÖR;

                             STEPAR jämfört med Holt-Winters: 12-månaders prognosjämförelse                             


BY Group: line_of_business=Bil

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 48
Forecast Periods: 6


BY Group: line_of_business=Hem

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 48
Forecast Periods: 6



                                          6-månaders prognoser per affärsgren                                           

  Obs   Affärsgren   DATE    _TYPE_   Prognostiserade skadeanmälningar  L95_CLAIM_COUNT  U95_CLAIM_COUNT  RESIDUAL_CLAIM_COUNT
    1  Bil          23742  FORECAST                        2524.596717      2359.095846      2690.097589                     .
    2  Bil          23773  FORECAST                        2604.418724      2370.365147        2838.4723                     .
    3  Bil          23801  FORECAST                        2576.092313      2289.436395       2


NOTE: DATA multi_book


NOTE: Wrote multi_book (96 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC SORT data=multi_book

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 96 rows from multi_book.
NOTE: Wrote multi_book (96 rows, 3 columns).
NOTE: PROC SORT statement used.
NOTE: PROC FORECAST data=multi_book

NOTE: BY Group: line_of_business=Bil
NOTE: Using Python for FORECAST estimation
NOTE: BY Group: line_of_business=Hem
NOTE: Using Python for FORECAST estimation
NOTE: Output dataset created with 108 observations.
NOTE: PROC PRINT data=fc_by

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: PROC PRINT completed: 6 observations printed, 7 variables


## Att tolka resultaten

**Vad modellerna säger reserveringsteamet:**

- **STEPAR** återger den uppåtgående driften och den kortsiktiga momentumen, men dess 12-månaders prognos avklingar jämnt från cirka 2 620 (steg 1) mot ungefär 1 980 vid mitten av horisonten innan den driver tillbaka till cirka 2 140 — den återger inte en skarp vintertopp, eftersom den bara hanterar säsongsvariation via autoregressiva lagg. Den är en rimlig snabb baslinje.
- **WINTERS** med `SEASONS=12` för direkt in den årliga rytmen via sina multiplikativa säsongsfaktorer: dess prognos är högst tidigt i horisonten (cirka 2 780 vid steg 1, cirka 2 900 vid steg 2), avtar till en svacka nära steg 9 (cirka 2 200), och stiger igen vid steg 12 (cirka 2 800). Jämfört med STEPAR-baslinjen ligger den **150-660 skador högre** under merparten av horisonten (se kolumnen `Winters - STEPAR` i steg 4) — den skillnaden är den säsongssignal den autoregressiva modellen missar.
- **95 %-konfidensbandet** (kolumnerna `L95_*`/`U95_*`, styrda av `ALPHA=`) vidgas i takt med att horisonten förlängs — för WINTERS-modellen från en bredd på ungefär 410 skador vid steg 1 till cirka 1 420 vid steg 12 — den ärliga signalen att skattningar sent i horisonten bär mer osäkerhet än skattningar nära i tiden. Reservsättare bör hålla kapital mot den övre gränsen, inte bara punktprognosen.
- **BY-bearbetning** producerar Bil- och Hem-prognoserna i en enda körning, var och en med sin egen säsongsanpassning. Bil-beståndet prognostiserar i intervallet cirka 2 510-2 600 medan Hem-beståndet ligger nära 1 870-2 030, så varje gren behåller sin egen nivå och säsongsform — mönstret teamet skulle utvidga till varje produkt i portföljen.

**Slutsats:** för en skadeantalsserie med en tydlig årscykel är `METHOD=WINTERS SEASONS=12` det idiomatiska valet; STEPAR-baslinjen är en användbar rimlighetskontroll, och `OUTFULL`/`OUTLIMIT` plus en steg-för-steg-modelljämförelse låter aktuarien dimensionera den framåtblickande reserven med sitt osäkerhetsband.

> **Motortrohetsanmärkning.** Denna notebook dokumenterar två nuvarande begränsningar i Jenners PROC FORECAST (gap-test `400979`): prognoshorisontens `ID` stegas fram med en enhet per steg i stället för via `INTERVAL=MONTH`, så de utskrivna prognosdatumen är inte de avsedda 2025-kalendermånaderna (vi granskar i stället horisonten per steg); och `OUTRESID`/`OUTALL` fyller ännu inte i ett-steg-framåt-`_TYPE_='RESIDUAL'`-raderna, så residualdiagnostik ersätts med en direkt tvåmodellsjämförelse. Själva punktprognoserna och konfidensgränserna produceras och är det narrativet ovan grundar sig på.